# 거래 가능 나스닥 종목 30년치 가격 (yfinance)
- 종목 목록: `tradable_assets.md`의 **나스닥 (USD)** 섹션
- 기간: 1996-01-01 ~ 최근 (코스피 데이터와 같은 시작일)
- `auto_adjust=False`로 받아서 두 가지를 저장
  - `Close`: 야후 종가. ⚠️ **액면분할은 반영**되어 있고 배당은 미반영 (DataGuide 원주가와 다름)
  - `Adj Close`: 분할 + 배당 반영 → **수익률 계산용** (DataGuide 수정주가에 대응)
- 원주가(`raw_close`): 야후는 직접 제공하지 않으므로 `Close × (이후 분할비율 누적곱)`으로 복원
  → 주문 수량·체결가 시뮬레이션용 (수익률 계산에 쓰면 분할일에 가짜 급락 발생)

In [ ]:
import re
from pathlib import Path
import pandas as pd
import yfinance as yf

PROJECT_DIR = Path("/Users/kasia/Desktop/학교/26-2/금프2/프로젝트")
MD_PATH = PROJECT_DIR / "tradable_assets.md"
START = "1996-01-01"

## 1. md 파일에서 나스닥 티커 추출

In [ ]:
text = MD_PATH.read_text(encoding="utf-8")
sec = re.search(r"^## 나스닥.*?$(.*?)(?=^## |\Z)", text, flags=re.M | re.S).group(1)

targets = {}
for item in sec.split("/"):
    m = re.search(r"\b([A-Z][A-Z.]{0,5})\s+([^/\n]+)", item)
    if m:
        targets[m.group(1)] = m.group(2).strip()
print(len(targets), "종목")
list(targets.items())[:5]

## 2. yfinance로 다운로드

In [ ]:
# actions=True: 액면분할(Stock Splits) 이력도 같이 받음
raw = yf.download(list(targets), start=START, auto_adjust=False,
                  actions=True, progress=False, threads=True)
close = raw["Close"].dropna(how="all")
adj   = raw["Adj Close"].dropna(how="all")
close.index.name = adj.index.name = "date"

failed = [t for t in targets if t not in adj.columns or adj[t].notna().sum() == 0]
ok = [t for t in targets if t not in failed]
close, adj = close[ok], adj[ok]
print("기간:", adj.index.min().date(), "~", adj.index.max().date(), "/", len(adj), "거래일")
print("받은 종목:", len(ok), "/", len(targets))
print("실패:", {t: targets[t] for t in failed})

## 2-1. 원주가 복원 (분할 되돌리기)
- 날짜 t의 원주가 = Close(t) × ∏(t 이후 발생한 분할 비율)
- 예: NVDA 2024-06-10 10:1 분할 → 그 이전 Close에 ×10 (그 이전 분할들도 누적)

In [ ]:
splits = raw["Stock Splits"][ok].reindex(close.index).fillna(0).replace(0, 1)   # 분할 없는 날 = 1
# t 이후(t 당일 제외) 분할 비율 누적곱
after = splits[::-1].cumprod()[::-1].shift(-1).fillna(1)
raw_close = close * after

n_split = (splits != 1).sum()
print("분할 이력 있는 종목:", (n_split > 0).sum(), "/", len(ok), "| 총 분할 횟수:", n_split.sum())
splits[splits != 1].stack().dropna().rename("ratio").sort_index().tail(10)

### 검증: 알려진 실제 종가와 비교

In [ ]:
known = [  # (티커, 날짜, 실제 당시 종가 USD) - 분할 직전 거래일
    ("AAPL", "2020-08-28", 499.23),   # 4:1 분할 직전
    ("TSLA", "2020-08-28", 2213.40),  # 5:1 분할 직전
    ("NVDA", "2024-06-07", 1208.88),  # 10:1 분할 직전
    ("AMZN", "2022-06-03", 2447.00),  # 20:1 분할 직전
]
pd.DataFrame([(t, d, raw_close.loc[d, t], v) for t, d, v in known],
             columns=["ticker", "date", "복원 원주가", "실제 종가"]).round(2)

## 3. 종목별 커버리지

In [ ]:
summary = pd.DataFrame({
    "md_name": [targets[t] for t in ok],
    "first_date": [adj[t].first_valid_index() for t in ok],
    "last_date":  [adj[t].last_valid_index() for t in ok],
    "n_obs": adj.notna().sum().values,
}, index=pd.Index(ok, name="ticker"))
summary["full_30y"] = summary["first_date"] <= adj.index[0]
print("30년 전체 데이터 보유:", summary["full_30y"].sum(), "종목")
summary.sort_values("first_date")

## 4. 검증: 이상 수익률 확인

In [ ]:
r = adj.pct_change(fill_method=None).stack()
r[r.abs() > 0.5].sort_values()   # 일간 ±50% 초과 (데이터 오류 여부 확인용)

## 5. 저장

In [ ]:
close.to_csv(PROJECT_DIR / "nasdaq_close_30y.csv")
adj.to_csv(PROJECT_DIR / "nasdaq_adj_close_30y.csv")
raw_close.to_csv(PROJECT_DIR / "nasdaq_raw_close_30y.csv")
splits[splits != 1].stack().dropna().rename("ratio").to_csv(PROJECT_DIR / "nasdaq_splits_30y.csv")
summary.to_csv(PROJECT_DIR / "nasdaq_30y_summary.csv", encoding="utf-8-sig")
print("saved:", close.shape, adj.shape, raw_close.shape)
adj.tail()